# 05: Dissertation results

The experiments of the MSc dissertation, rebuilt from its 16 trained checkpoints: which encoder and
decoder to use, how much KL regularisation, and how well the chosen model completes partial teeth.
Every number comes from the per-tooth test results stored in the checkpoints; they reproduce the
report's tables exactly.

**Needs** the checkpoints in `artifacts/dissertation_checkpoints/` (not in this repository). The 3D views
also need the dissertation's own processed data in `data/processed/complete/` and `partial/`, and are
skipped without it. Runs on a CPU in under a minute. Teeth3DS+ is licensed: **clear this notebook's
outputs before committing**, as the 3D views contain the data.

Every chart is interactive: hover for values, click legend entries to hide them.

In [ ]:
# Colab setup: fetch this repository's latest code from GitHub. Does nothing locally.
# Same cell in every notebook. After pushing new code: Runtime > Restart session, then run all cells.
import sys

REPO = "dgelbier23/dental-3d-vae"

if "google.colab" in sys.modules:
    import subprocess
    from pathlib import Path
    from google.colab import userdata

    try:  # A token is only needed while the repository is private.
        token = userdata.get("GITHUB_TOKEN")
    except Exception:  # No such secret, or this notebook may not read it.
        token = ""
    login = f"x-access-token:{token}@" if token else ""
    repo_dir = Path("/content") / REPO.split("/")[1]
    subprocess.run(["rm", "-rf", str(repo_dir)], check=True)
    result = subprocess.run(["git", "clone", "--depth", "1", f"https://{login}github.com/{REPO}.git", str(repo_dir)],
                            capture_output=True, text=True)
    if result.returncode:
        error = result.stderr.replace(token, "***") if token else result.stderr
        raise RuntimeError(f"{error} If the repository is private, add a GITHUB_TOKEN Colab secret (see the README).")
    if token:  # git clone saves the URL it was given, so remove the token from the clone's settings.
        subprocess.run(["git", "-C", str(repo_dir), "remote", "set-url", "origin",
                        f"https://github.com/{REPO}.git"], check=True)
    sys.path.insert(0, str(repo_dir / "src"))

## Setup

In [ ]:
import pandas as pd

from dental_3d_vae.experiments.dissertation import load_dissertation_run
from dental_3d_vae.ml.evaluate import summarise
from dental_3d_vae.runtime.env import REPO_ROOT, get_data_dir
from dental_3d_vae.viz import results as charts
from dental_3d_vae.viz.plotting import use_theme

use_theme()
CHECKPOINT_DIR = get_data_dir().parent / "artifacts" / "dissertation_checkpoints"
DATA_DIR = get_data_dir() / "processed"  # complete/ and partial/: the dissertation's own build
SAVE_FIGURES = False  # True, locally: save the README's figures (needs kaleido and Chrome).
FIGURES_DIR = REPO_ROOT / "docs" / "figures" / "results"
README_FIGURES = {"decoders_by_tooth", "beta", "missingness", "completion"}

# Run IDs, e.g. "D1" for COMPLETE_D1_dgcnn_foldingnet.pt.
runs = {path.stem.split("_")[1]: load_dissertation_run(path) for path in sorted(CHECKPOINT_DIR.glob("*.pt"))}
print(f"{len(runs)} runs: {', '.join(runs)}")


def results_of(group: dict[str, str]) -> dict:
    """{label: per-tooth test results} for {label: run ID}."""
    return {label: runs[run]["results"] for label, run in group.items()}


def summary_table(group: dict[str, str]) -> pd.DataFrame:
    """Test Chamfer distance per run, plus its size and training cost."""
    table = summarise(results_of(group))
    table["Parameters"] = [runs[run]["summary"]["num_parameters"] for run in group.values()]
    table["Minutes"] = [runs[run]["summary"]["training_minutes"] for run in group.values()]
    table["GPU"] = [runs[run]["summary"]["gpu"].replace("NVIDIA ", "") for run in group.values()]
    return table


def show(fig, name: str) -> None:
    """Show ``fig``; with SAVE_FIGURES, also save it if the README uses it."""
    fig.show()
    if SAVE_FIGURES and name in README_FIGURES:
        FIGURES_DIR.mkdir(parents=True, exist_ok=True)
        fig.write_image(FIGURES_DIR / f"{name}.png", width=1100, scale=2)

# Part 1: Complete-to-complete reconstruction

Each model learns to reproduce complete teeth: 2,826 for training, 606 for testing. Every run uses the
same settings (latent size 32, Adam at 1e-3, batch 32, early stopping after 20 epochs without a better
validation Chamfer distance), so each experiment changes one thing.

## 1.1 Encoders

Four encoders, each with the MLP decoder and no KL term (β = 0), all trained on the same T4 GPU.
The DGCNN run was saved as `COMPLETE_TEST_01` (run ID "TEST"); D0 is the same model retrained on an
A100 for the decoder experiment.

In [ ]:
ENCODERS = {"PointNet": "E0", "T-Net PointNet": "E1", "PointNet++": "E2", "DGCNN": "TEST"}
show(charts.plot_table(summary_table(ENCODERS).rename_axis("Encoder"), "Test Chamfer distance by encoder",
                       lowest_is_best=("Mean", "Median", "IQR", "Min", "Max", "Parameters")), "encoders_table")
encoder_histories = {label: runs[run]["metrics"] for label, run in ENCODERS.items()}
show(charts.plot_loss_curves(encoder_histories, "train_chamfer"), "encoders_training")
show(charts.plot_loss_curves(encoder_histories, "val_chamfer"), "encoders_validation")

The four encoders end within about 5% of each other in mean test Chamfer distance, a gap similar to
the variation between reruns of one configuration. DGCNN was carried forward for its fast early
learning and its explicit modelling of local neighbourhoods, which matters more once part of the tooth
is missing.

## 1.2 Decoders

DGCNN with each decoder, β = 0.

In [ ]:
DECODERS = {"MLP": "D0", "FoldingNet": "D1", "AtlasNet": "D2"}
show(charts.plot_table(summary_table(DECODERS).rename_axis("Decoder"), "Test Chamfer distance by decoder",
                       lowest_is_best=("Mean", "Median", "IQR", "Min", "Max", "Parameters")), "decoders_table")
show(charts.plot_distributions(results_of(DECODERS)), "decoders_distribution")
show(charts.plot_odontogram(results_of(DECODERS)), "decoders_by_tooth")

FoldingNet is the most accurate (mean Chamfer distance 14% below the MLP's) with 91% fewer parameters,
and is the best of the three on 24 of the 32 tooth positions. For every decoder, lower incisors are the
easiest teeth and third molars the hardest.

Below, three teeth (each the median-error tooth of its type for FoldingNet), as each decoder rebuilds
them. The MLP predicts every point independently and scatters them; FoldingNet and AtlasNet deform flat
grids, so their points form coherent surfaces.

In [ ]:
import torch

from dental_3d_vae.ml.data import CompleteToothDataset
from dental_3d_vae.ml.metrics import chamfer_distance
from dental_3d_vae.ml.models.vae import load_checkpoint
from dental_3d_vae.viz.pointclouds import plot_pointclouds


def median_tooth(results: pd.DataFrame) -> int:
    """Row label of the median-Chamfer-distance tooth in ``results``."""
    return results.sort_values("cd").index[len(results) // 2]


if (DATA_DIR / "complete").exists():
    test = runs["D1"]["results"]
    picks = [median_tooth(test[test["tooth_type"] == kind])
             for kind in ("Lower left central incisor", "Upper right canine", "Upper left first molar")]
    teeth = CompleteToothDataset(DATA_DIR / "complete" / "teeth_dataset.h5", test.loc[picks])
    models = {label: load_checkpoint(next(CHECKPOINT_DIR.glob(f"*_{run}_*.pt")))[0] for label, run in DECODERS.items()}
    clouds, titles = [], []
    with torch.no_grad():
        for tooth in teeth:
            clouds.append(tooth["complete"])
            titles.append(f"<b>{tooth['tooth_id']}</b> ground truth")
            for label, model in models.items():
                reconstruction = model.reconstruct(tooth["complete"][None])
                clouds.append(reconstruction[0])
                titles.append(f"{label}: CD {chamfer_distance(reconstruction, tooth['complete'][None]).item():.4f}")
    plot_pointclouds(clouds, titles, cols=4).show()
else:
    print(f"No processed data in {DATA_DIR / 'complete'}: 3D comparison skipped.")

## 1.3 KL weight β

DGCNN + FoldingNet with eight KL weights. β trades reconstruction accuracy for a smoother, more regular
latent space: the regularity that makes sampling several completions meaningful.

In [ ]:
BETAS = {0: "D1", 1e-5: "B1", 1e-4: "B2", 1e-3: "B3", 1e-2: "B4", 0.05: "B5", 0.1: "B6", 0.25: "B7"}
show(charts.plot_beta_ablation({beta: runs[run]["results"] for beta, run in BETAS.items()},
                               {beta: runs[run]["summary"]["test_kl"] for beta, run in BETAS.items()},
                               selected=1e-3), "beta")
beta_table = summarise({f"{beta:g}": runs[run]["results"] for beta, run in BETAS.items()}).rename_axis("β")
beta_table["KL"] = [runs[run]["summary"]["test_kl"] for run in BETAS.values()]
show(charts.plot_table(beta_table, "Test Chamfer distance and KL divergence by β",
                       lowest_is_best=("Mean", "Median", "IQR", "Min")), "beta_table")

β = 0.001 keeps reconstruction close to the unregularised model (mean Chamfer distance 0.0622 against
0.0600) while more than halving the KL divergence, so it was chosen for the partial-to-complete models.
Beyond 0.01, accuracy falls quickly.

# Part 2: Partial-to-complete reconstruction

DGCNN + FoldingNet, β = 0.001, trained to rebuild the complete tooth from a partial one. One model per
removal method: **plane cut** (P0) and **connected patch** (P1), each seeing 0, 15, 30 and 45% of the
points removed. Test set: 606 teeth × 4 levels = 2,424 partial teeth per model.

ORPE here is the dissertation's stored value, measured against the upsampled model input; this
repository now measures it on the observed points (0.3-0.4% higher).

In [ ]:
PARTIAL = {"Plane cut": "P0", "Connected patch": "P1"}
overall = pd.DataFrame({label: frame[["cd", "mrre", "orpe"]].mean() for label, frame in results_of(PARTIAL).items()}).T
overall.columns = ["Mean CD", "Mean MRRE", "Mean ORPE"]
show(charts.plot_table(overall.rename_axis("Removal"), "Partial-to-complete test results",
                       lowest_is_best=tuple(overall.columns)), "partial_table")
show(charts.plot_distributions(results_of(PARTIAL)), "partial_distribution")

## 2.1 Effect of missing geometry

In [ ]:
show(charts.plot_metrics_vs_missingness(results_of(PARTIAL)), "missingness")
plane = runs["P0"]["results"]
show(charts.plot_odontogram({f"{fraction:.0%} removed": plane[plane["missing_fraction"] == fraction]
                             for fraction in (0, 0.15, 0.3, 0.45)}, mark_best=False), "plane_by_tooth")

Error rises steadily with the amount removed: from 15% to 45% missing, mean Chamfer distance rises by
12% for plane cuts and 10% for connected patches. The rise concentrates in the hardest teeth, the third
molars. ORPE barely moves: the model keeps what it was shown even when much of the tooth is missing.

## 2.2 Global against local error

Chamfer distance scores the whole tooth; MRRE only the removed part, ORPE only the observed part. If
they measured the same thing, the points below would lie on a line.

In [ ]:
show(charts.plot_metric_relationship(results_of(PARTIAL), "cd", "mrre"), "cd_vs_mrre")
show(charts.plot_metric_relationship(results_of(PARTIAL), "cd", "orpe"), "cd_vs_orpe")
show(charts.plot_metric_relationship(results_of(PARTIAL), "mrre", "orpe"), "mrre_vs_orpe")

Chamfer distance tracks both local errors (r = 0.78-0.90) but not perfectly: a good overall score can
hide a poorly rebuilt missing region, which is why the local metrics are reported too.

## 2.3 Completions, close up

The plane-cut model with 45% of the tooth removed, on two test teeth: a typical one (the median error
over all teeth) and one of its best molars (the second-lowest error among molars: the clearest molar
shape of the top three, and the README's opening image). For each: what it was given, what it built,
and where it misses.

In [ ]:
from dental_3d_vae.ml.data import PartialToothDataset
from dental_3d_vae.viz.pointclouds import plot_completion, plot_point_errors, save_rotating_gif

if (DATA_DIR / "partial" / "partial_masks.h5").exists():
    cuts = plane[plane["missing_fraction"] == 0.45]
    molars = cuts[cuts["tooth_type"].str.contains(" molar")]  # The space excludes premolars.
    examples = {"A typical tooth (median error)": median_tooth(cuts),
                "One of the best molars": molars.sort_values("cd").index[1]}
    partials = PartialToothDataset(DATA_DIR / "complete" / "teeth_dataset.h5", DATA_DIR / "partial" / "partial_masks.h5",
                                   plane.loc[list(examples.values())])
    model, _ = load_checkpoint(next(CHECKPOINT_DIR.glob("*_P0_*.pt")))
    for (label, row), tooth in zip(examples.items(), partials):
        with torch.no_grad():
            reconstruction = model.reconstruct(tooth["partial"][None])[0]
        print(f"{label}: tooth {tooth['tooth_id']} ({plane.loc[row, 'tooth_type']}), CD {plane.loc[row, 'cd']:.4f}"
              f" (median for molars: {molars['cd'].median():.4f})")
        completion_fig = plot_completion(tooth["complete"], tooth["removed"], reconstruction)
        completion_fig.show()
        if SAVE_FIGURES and "completion" in README_FIGURES and label.startswith("One of the best"):
            # Without the ground truth until the dataset licence is confirmed.
            save_rotating_gif(plot_completion(tooth["complete"], tooth["removed"], reconstruction, ground_truth=False),
                              FIGURES_DIR / "completion.gif", width=900, height=380)
        plot_point_errors(tooth["complete"], reconstruction, tooth["removed"]).show()
else:
    print(f"No dissertation partials in {DATA_DIR / 'partial'}: completion views skipped.")

## 2.4 Stochastic consistency

Decoding several samples from the latent distribution gives several plausible completions; the
Stochastic Consistency Error (SCE) is their mean pairwise Chamfer distance. SCE was not stored in the
checkpoints, so it comes from `04_evaluate_run.ipynb` (on a GPU), which saves
`<checkpoint>_test_results.csv` next to each checkpoint. This section shows it once those files exist.

In [ ]:
sce_results = {label: pd.read_csv(path) for label, run in PARTIAL.items()
               for path in CHECKPOINT_DIR.glob(f"*_{run}_*_test_results.csv")}
if len(sce_results) == len(PARTIAL):
    show(charts.plot_distributions(sce_results, "sce"), "sce_distribution")
    show(charts.plot_metric_relationship(sce_results, "cd", "sce"), "cd_vs_sce")
else:
    print("No SCE results yet: evaluate P0 and P1 with 04_evaluate_run.ipynb on a GPU.")

The dissertation found mean SCE of 0.0313 (plane) and 0.0314 (patch) and only weak correlation with
Chamfer distance (r = 0.29 and 0.20): the model agreeing with itself says little about whether it is
right, so consistency cannot stand in for accuracy.